In [1]:
# =============================================================================
# 4-METHODS COMPARISON — redshift removed (observed frame)
# M2 raw high-freq: fine-grid median (continuous)
# M3/M4 raw high-freq: sparse season DCT points (scatter)
# Legend in ensembles shows α_high and ν_b per bin
# LaTeX labels throughout; colors with better contrast
# =============================================================================

import os, glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib import patheffects
from scipy.optimize import curve_fit
from scipy.fft import dct
from astropy.timeseries import LombScargle
import warnings
warnings.filterwarnings('ignore')

# ----------------------------- CONFIG ---------------------------------------
BASE_DIR = r"ztffp_final_data_2025"
CSV_META = r"Cross_15_luiz.csv"
BAND = 'g'
PLOTS_DIR = f"plots_4methods_{BAND}"
os.makedirs(PLOTS_DIR, exist_ok=True)
REPORT_PATH = os.path.join(PLOTS_DIR, "report_4methods.txt")

# Observed frame (no cosmological redshift correction)
CADENCE_DAYS = 1.0
F_MAX_NYQUIST = 1.0 / (2.0 * CADENCE_DAYS)   # = 0.5 d^-1
SEASON_GAP_DAYS = 30.0
MIN_SEASON_POINTS = 5
Z_MIN, Z_MAX = 0.4, 0.5                       # filter only; not used for time
TARGET_DEX_STEP = 0.10
N_FINE_GRID = 300

# --- Styling ---
BAND_ALPHA = 0.25
RAW_LW = 1.0
RAW_ALPHA = 0.80
SPARSE_ALPHA = 0.35
SPARSE_SIZE = 4
REF_LW = 1.0
REF_ALPHA = 0.80
FIT_LW = 2.0
MIN_BAND_N = 30
BAND_FLOOR = 0.10

mass_edges = [7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0]
edd_edges = [-2.5, -2.0, -1.5, -1.0, -0.5, 0.0]

TMAX_DIAGONALS = []
for k_const in range(-(len(edd_edges) - 1), len(mass_edges)):
    _cells = []
    for _row in range(len(edd_edges) - 1):
        _col = _row + k_const
        if 0 <= _col < len(mass_edges) - 1:
            _cells.append((_row, _col))
    if len(_cells) < 2:
        continue
    _rows = [r for r, c in _cells]
    _cols = [c for r, c in _cells]
    TMAX_DIAGONALS.append({
        'row_range': (min(_rows), max(_rows)),
        'col_range': (min(_cols), max(_cols)),
        'k_const': k_const,
    })

MIN_OBJS_PER_BIN = 30
MIN_CONTRIB_LEAKAGE = 10
MIN_CONTRIB_ENSEMBLE = 10
NU_B_MAX = 0.5
FLUX_ERR_FACTOR = 2.0
METHOD_DT_GRID = 30.0

METHODS = ['m1_LS_full', 'm2_intra_LS', 'm3_intra_DCT', 'm4_intra_LSmeans']
METHOD_LABELS = {'m1_LS_full': 'M1: LS full curve',
                 'm2_intra_LS': 'M2: Intra Season(high) + LS(low)',
                 'm3_intra_DCT': 'M3: Intra Season(high) + DCT(means)',
                 'm4_intra_LSmeans': 'M4: Intra Season(high) + LS(means)'}
COLORS = {'m1_LS_full': 'tab:blue',
          'm2_intra_LS': 'tab:red',
          'm3_intra_DCT': 'tab:orange',
          'm4_intra_LSmeans': 'tab:purple'}
HIGH_FREQ_SPARSE = {'m3_intra_DCT', 'm4_intra_LSmeans'}

LIT = {
    'Petrecca+2026 (BPL, a_low=-1)': {'nu_b': 1e-3, 'a_high': -2.7, 'a_high_err': 0.1},
    'Stone+2022 (2nd break)':        {'nu_b': 3e-2, 'a_high': None, 'a_high_err': None},
}

# ----------------------------- AUXILIARY ------------------------------------
def coadd_nightly(t_obs, f_obs, f_err):
    day_ids = np.floor(t_obs).astype(int)
    unique_days = np.unique(day_ids)
    t_d, f_d, e_d = [], [], []
    for d in unique_days:
        m = (day_ids == d)
        if np.sum(m) == 1:
            t_d.append(t_obs[m][0]); f_d.append(f_obs[m][0]); e_d.append(f_err[m][0])
        else:
            w = 1.0/(f_err[m]**2 + 1e-30)
            t_d.append(np.mean(t_obs[m]))
            f_d.append(np.sum(w*f_obs[m])/np.sum(w))
            e_d.append(1.0/np.sqrt(np.sum(w)))
    return np.array(t_d), np.array(f_d), np.array(e_d)

def bin_to_log_grid(freqs, powers, edges):
    n = len(edges) - 1
    b = np.full(n, np.nan)
    idx = np.searchsorted(edges, freqs, side='right') - 1
    v = (idx >= 0) & (idx < n) & np.isfinite(powers) & (powers > 0)
    if not np.any(v):
        return b
    sums = np.bincount(idx[v], weights=powers[v], minlength=n)
    cnts = np.bincount(idx[v], minlength=n)
    m = cnts > 0
    b[m] = sums[m]/cnts[m]
    return b

def ls_psd(t, f, f_min, f_max):
    ls = LombScargle(t, f, fit_mean=True, center_data=True)
    fr, p = ls.autopower(normalization='psd', minimum_frequency=f_min,
                          maximum_frequency=f_max, samples_per_peak=5)
    vt = float(np.var(f))
    if len(fr) > 1:
        integ = float(np.trapz(p, fr))
        if integ > 0 and np.isfinite(integ):
            p = p * (vt/integ)
    return fr, p

def dct_psd(t, f, cadence, var_target=None):
    if len(t) < 4:
        return np.array([]), np.array([])
    n = len(f)
    coeffs = dct(f, type=2, norm='ortho')
    freqs = np.arange(1, n)/(2.0*n*cadence)
    v = freqs <= (1.0/(2.0*cadence))
    freqs = freqs[v]
    power = (coeffs[1:][v]**2) * (2.0*cadence)
    if var_target is None:
        var_target = float(np.var(f))
    if len(freqs) > 1:
        integ = float(np.trapz(power, freqs))
        if integ > 0 and np.isfinite(integ):
            power = power * (var_target/integ)
    return freqs, power

def get_sparse_high_freq(seasons, f_cut):
    """Concatenate all season DCT points with f > f_cut (raw, unbinned)."""
    f_all, p_all = [], []
    for s in seasons:
        fr = s['fr_dct']; pw = s['pw_dct']
        v = (fr > f_cut) & (pw > 0) & np.isfinite(fr) & np.isfinite(pw)
        if np.sum(v) > 0:
            f_all.append(fr[v]); p_all.append(pw[v])
    if len(f_all) == 0:
        return np.array([]), np.array([])
    return np.concatenate(f_all), np.concatenate(p_all)

def get_fine_grid_median(seasons, f_grid, f_min_use):
    """Interpolate each season DCT to f_grid, take median -> continuous."""
    interp = []
    for s in seasons:
        fr = s['fr_dct']; pw = s['pw_dct']
        v = (fr > 0) & (pw > 0) & np.isfinite(fr) & np.isfinite(pw)
        if np.sum(v) < 3:
            continue
        try:
            lp = np.interp(np.log10(f_grid), np.log10(fr[v]), np.log10(pw[v]),
                           left=np.nan, right=np.nan)
            interp.append(lp)
        except Exception:
            continue
    if len(interp) == 0:
        return np.array([]), np.array([])
    med = np.nanmedian(np.array(interp), axis=0)
    p = 10**med
    valid = np.isfinite(p) & (p > 0) & (f_grid > f_min_use)
    return f_grid[valid], p[valid]

def draw_band_and_markers(ax, x, med, has_band, lower, upper, color, label,
                          ms=6, ms_open=7):
    ax.loglog(x, med, '-', color=color, lw=1.5, alpha=0.85, label=label, zorder=2)
    if np.any(has_band):
        ax.loglog(x[has_band], med[has_band], 'o', color=color, ms=ms, zorder=3)
    if np.any(~has_band):
        ax.loglog(x[~has_band], med[~has_band], 'o', ms=ms_open,
                  markerfacecolor='white', markeredgecolor=color,
                  markeredgewidth=1.5, zorder=5)
    if np.any(has_band):
        ax.fill_between(x[has_band], lower[has_band], upper[has_band],
                        color=color, alpha=BAND_ALPHA, zorder=1)

# ----------------------------- BPL MODEL ------------------------------------
def model_bpl(f, norm, nu_b, a_high, noise):
    return norm * f**(-1.0) * (1.0 + (f/nu_b)**(-1.0 - a_high))**(-1.0) + noise

def fit_bpl(freq, power, sigma=None, nu_b_min=None, nu_b_max=NU_B_MAX):
    mask = (power > 0) & np.isfinite(power) & (freq > 0)
    if sigma is not None:
        mask &= (sigma > 0) & np.isfinite(sigma)
    if np.sum(mask) < 5:
        return None
    f_fit, p_fit = freq[mask], power[mask]
    s = sigma[mask] if sigma is not None else None
    if nu_b_min is None:
        nu_b_min = float(f_fit.min())
    p0 = [np.median(p_fit*f_fit)*10, np.sqrt(nu_b_min*nu_b_max), -2.7, 0.1*np.min(p_fit)]
    bnds = ([1e-12, nu_b_min, -5.0, 0.0], [1e3, nu_b_max, -1.0, np.max(p_fit)*2])
    try:
        popt, pcov = curve_fit(model_bpl, f_fit, p_fit, p0=p0, sigma=s,
                                absolute_sigma=False, bounds=bnds, maxfev=10000)
        perr = np.sqrt(np.diag(pcov))
        if s is not None:
            chi2 = float(np.sum(((p_fit - model_bpl(f_fit, *popt))/s)**2))
        else:
            chi2 = float(np.sum((p_fit - model_bpl(f_fit, *popt))**2))
        Np = int(np.sum(mask))
        bic = chi2 + 4*np.log(Np)
        ab = (popt[1] <= nu_b_min*1.01) or (popt[1] >= nu_b_max*0.99)
        wb = 'lower' if popt[1] <= nu_b_min*1.01 else ('upper' if popt[1] >= nu_b_max*0.99 else None)
        return {'norm': popt[0], 'norm_err': perr[0],
                'nu_b': popt[1], 'nu_b_err': perr[1],
                'alpha_high': popt[2], 'alpha_high_err': perr[2],
                'noise': popt[3], 'noise_err': perr[3],
                'at_bound': ab, 'which_bound': wb,
                'chi2': chi2, 'N_pts': Np, 'bic': bic}
    except Exception:
        return None

def diag_label(k_const):
    cells = []
    for row in range(len(edd_edges) - 1):
        col = row + k_const
        if 0 <= col < len(mass_edges) - 1:
            cells.append((row, col))
    if not cells:
        return f'col-row={k_const:+d}'
    logTs = []
    for row, col in cells:
        x_c = (mass_edges[col] + mass_edges[col+1]) / 2
        y_c = (edd_edges[row] + edd_edges[row+1]) / 2
        logTs.append(0.25 * (y_c - x_c))
    return f'$\\log T\\approx{float(np.median(logTs)):.2f}$'

def cells_on_diagonal(d):
    k = d['k_const']
    cells = []
    for row in range(d['row_range'][0], d['row_range'][1] + 1):
        col = row + k
        if 0 <= col < len(mass_edges) - 1 and 0 <= row < len(edd_edges) - 1:
            cells.append((row, col))
    return cells

def mask_for_cells(cells):
    mask = np.zeros_like(mi_all, dtype=bool)
    for row, col in cells:
        mask |= (mi_all == col) & (ei_all == row)
    return mask

def legend_with_fit(base_label, N, fit):
    """Legend text with LaTeX alpha_high and nu_b."""
    if fit is None:
        return f"{base_label} (N={N})"
    lbl = (f"{base_label} (N={N}): "
           f"$\\alpha_{{\\rm high}}$={fit['alpha_high']:.2f}"
           f"$\\pm${fit['alpha_high_err']:.2f}, "
           f"$\\nu_b$={fit['nu_b']:.2e}")
    if fit['at_bound']:
        lbl += f" [{fit['which_bound']}]"
    return lbl

# ----------------------------- LOAD + CROSSMATCH ----------------------------
print("Loading metadata...")
meta = pd.read_csv(CSV_META)
ra_cat = meta['RA'].values; dec_cat = meta['DEC'].values
logmbh_cat = meta['LOGMBH'].values; logedd_cat = meta['LOGLEDD_RATIO'].values
z_col = 'Z_DR16Q' if 'Z_DR16Q' in meta.columns else 'Z'
z_cat = meta[z_col].values

folders = [f for f in os.listdir(BASE_DIR)
           if os.path.isdir(os.path.join(BASE_DIR, f)) and "plots" not in f]

matched = []
for folder in tqdm(folders, desc="Crossmatch"):
    try:
        ra_f, dec_f = map(float, folder.split('_'))
    except Exception:
        continue
    dist = np.sqrt((ra_cat - ra_f)**2 + (dec_cat - dec_f)**2)
    idx = np.argmin(dist)
    if dist[idx] < 1e-4:
        z, m, e = z_cat[idx], logmbh_cat[idx], logedd_cat[idx]
        if np.isnan(m) or np.isnan(e) or np.isnan(z):
            continue
        if not (Z_MIN <= z <= Z_MAX):
            continue
        txts = glob.glob(os.path.join(BASE_DIR, folder, f"*_ztf_{BAND}_filtered.txt"))
        if txts:
            matched.append((folder, idx, txts[0]))
print(f"Matched: {len(matched)}")

# ----------------------------- PROCESS (no redshift) ------------------------
def process_object(t_raw, f_raw, err_raw):
    keep = err_raw <= FLUX_ERR_FACTOR * np.abs(f_raw)
    t_raw, f_raw, err_raw = t_raw[keep], f_raw[keep], err_raw[keep]
    if len(t_raw) < 20:
        return None
    t_obs, f_obs, f_err = coadd_nightly(t_raw, f_raw, err_raw)
    if len(t_obs) < 15:
        return None
    # --- observed frame: no (1+z) division ---
    t_rest = t_obs - t_obs.min()
    T_total = t_rest.max() - t_rest.min()
    f_med = np.median(f_obs)
    f_norm = (f_obs - f_med)/f_med
    breaks = np.where(np.diff(t_obs) > SEASON_GAP_DAYS)[0] + 1
    season_idx = np.split(np.arange(len(t_rest)), breaks)
    seasons = []
    for s in season_idx:
        if len(s) < MIN_SEASON_POINTS:
            continue
        t_s, f_s = t_rest[s], f_norm[s]
        n_g = int(np.floor((t_s.max() - t_s.min())/CADENCE_DAYS)) + 1
        if n_g < 4:
            continue
        t_grid = t_s.min() + np.arange(n_g)*CADENCE_DAYS
        f_grid = np.interp(t_grid, t_s, f_s)
        f_grid = f_grid - np.mean(f_grid)
        fr_d, pw_d = dct_psd(t_grid, f_grid, CADENCE_DAYS)
        seasons.append({
            't_s': t_s, 'f_s': f_s, 't_grid': t_grid, 'f_grid': f_grid,
            'fr_dct': fr_d, 'pw_dct': pw_d,
            't_mean_actual': float(np.mean(t_s)),
            'f_mean': float(np.mean(f_s)),
        })
    if len(seasons) < 2:
        return None
    N = len(seasons)
    t_means_eq = np.linspace(0, T_total, N)
    f_means = np.array([s['f_mean'] for s in seasons])
    return {
        't_rest': t_rest, 'f_norm': f_norm, 'T_total': T_total,
        'seasons': seasons, 't_means_eq': t_means_eq, 'f_means': f_means,
        't_obs': t_obs, 'f_obs': f_obs, 'n_seasons': N,
    }

valid_data = []
for folder, idx, txt_path in tqdm(matched, desc="Processing"):
    try:
        df = pd.read_csv(txt_path).sort_values('mjd')
        if len(df) < 20:
            continue
        t_raw = df['mjd'].values; f_raw = df['flux'].values
        err_raw = df['fluxe'].values if 'fluxe' in df.columns else np.zeros_like(f_raw)
        p = process_object(t_raw, f_raw, err_raw)
        if p is None:
            continue
        p['folder'] = folder
        p['LOGM'] = logmbh_cat[idx]; p['LOGE'] = logedd_cat[idx]
        p['z'] = z_cat[idx]   # stored for reference only, NOT used in time
        valid_data.append(p)
    except Exception:
        continue
print(f"Valid: {len(valid_data)}")

# ----------------------------- GLOBAL GRID ----------------------------------
f_min_g = 1.0/max(d['T_total'] for d in valid_data)
log_span = np.log10(F_MAX_NYQUIST) - np.log10(f_min_g)
n_bins = max(10, int(np.round(log_span/TARGET_DEX_STEP)))
bin_edges = np.logspace(np.log10(f_min_g), np.log10(F_MAX_NYQUIST), n_bins + 1)
bin_centers = np.sqrt(bin_edges[:-1]*bin_edges[1:])
f_fine = np.logspace(np.log10(f_min_g), np.log10(F_MAX_NYQUIST), N_FINE_GRID)
print(f"Grid: {n_bins} bins, [{f_min_g:.3e}, {F_MAX_NYQUIST:.3f}] d^-1 (OBSERVED frame)")
print(f"Fine grid: {N_FINE_GRID} pts")

# ----------------------------- COMPUTE ALL METHODS --------------------------
psd_mats = {m: np.full((len(valid_data), n_bins), np.nan) for m in METHODS}
raw_low = {m: [] for m in METHODS}
raw_high = {m: [] for m in METHODS}   # for M2: fine grid; M3/M4: sparse
f_cut_arr = np.full(len(valid_data), np.nan)

for i, d in enumerate(tqdm(valid_data, desc="Methods")):
    season_bins = []
    for s in d['seasons']:
        season_bins.append(bin_to_log_grid(s['fr_dct'], s['pw_dct'], bin_edges))
    season_mat = np.vstack(season_bins)
    intra_med = np.nanmedian(season_mat, axis=0)
    valid_hi = np.where(np.isfinite(intra_med))[0]
    if len(valid_hi) == 0:
        for m in METHODS:
            raw_low[m].append((np.array([]), np.array([])))
            raw_high[m].append((np.array([]), np.array([])))
        continue
    first_hi = valid_hi[0]
    f_cut = bin_edges[first_hi]
    f_cut_arr[i] = f_cut
    low_bin_mask = bin_centers < f_cut

    # M1: LS full
    f_ls_m1, p_ls_m1 = None, None
    try:
        f_ls_m1, p_ls_m1 = ls_psd(d['t_rest'], d['f_norm'],
                                   1.0/d['T_total'], F_MAX_NYQUIST)
        psd_mats['m1_LS_full'][i, :] = bin_to_log_grid(f_ls_m1, p_ls_m1, bin_edges)
        raw_low['m1_LS_full'].append((f_ls_m1, p_ls_m1))
    except Exception:
        raw_low['m1_LS_full'].append((np.array([]), np.array([])))
    raw_high['m1_LS_full'].append((np.array([]), np.array([])))

    # M2: Intra + LS low (mask M1, no renormalization)
    try:
        if f_ls_m1 is not None and len(f_ls_m1) > 0:
            low_f = f_ls_m1 < f_cut
            f_ls_low = f_ls_m1[low_f]; p_ls_low = p_ls_m1[low_f]
        else:
            f_ls_low, p_ls_low = np.array([]), np.array([])
        low_bin = bin_to_log_grid(f_ls_low, p_ls_low, bin_edges)
        combined = intra_med.copy()
        combined[low_bin_mask] = low_bin[low_bin_mask]
        psd_mats['m2_intra_LS'][i, :] = combined
        raw_low['m2_intra_LS'].append((f_ls_low, p_ls_low))
        f_fg, p_fg = get_fine_grid_median(d['seasons'], f_fine, f_cut)
        raw_high['m2_intra_LS'].append((f_fg, p_fg))
    except Exception:
        raw_low['m2_intra_LS'].append((np.array([]), np.array([])))
        raw_high['m2_intra_LS'].append((np.array([]), np.array([])))

    # M3: Intra + DCT of season means; sparse high-freq
    try:
        t_m = d['t_means_eq']; f_m = d['f_means']
        t_min, t_max = t_m.min(), t_m.max()
        n_g = int((t_max - t_min)/METHOD_DT_GRID) + 1
        if n_g >= 4:
            t_g = t_min + np.arange(n_g)*METHOD_DT_GRID
            f_int = np.interp(t_g, t_m, f_m)
            f_int = f_int - np.mean(f_int)
            f_d, p_d = dct_psd(t_g, f_int, METHOD_DT_GRID,
                                var_target=float(np.var(f_m)))
            low_bin = bin_to_log_grid(f_d, p_d, bin_edges)
            combined = intra_med.copy()
            combined[low_bin_mask] = low_bin[low_bin_mask]
            psd_mats['m3_intra_DCT'][i, :] = combined
            raw_low['m3_intra_DCT'].append((f_d, p_d))
            f_sp, p_sp = get_sparse_high_freq(d['seasons'], f_cut)
            raw_high['m3_intra_DCT'].append((f_sp, p_sp))
        else:
            raw_low['m3_intra_DCT'].append((np.array([]), np.array([])))
            raw_high['m3_intra_DCT'].append((np.array([]), np.array([])))
    except Exception:
        raw_low['m3_intra_DCT'].append((np.array([]), np.array([])))
        raw_high['m3_intra_DCT'].append((np.array([]), np.array([])))

    # M4: Intra + LS of season means; sparse high-freq
    try:
        t_m = d['t_means_eq']; f_m = d['f_means']
        if len(t_m) >= 3:
            ls_m = LombScargle(t_m, f_m, fit_mean=True, center_data=True)
            fr_m, p_m = ls_m.autopower(normalization='psd',
                                        minimum_frequency=1.0/d['T_total'],
                                        maximum_frequency=f_cut,
                                        samples_per_peak=5)
            vt = float(np.var(f_m))
            if len(fr_m) > 1:
                integ = float(np.trapz(p_m, fr_m))
                if integ > 0 and np.isfinite(integ):
                    p_m = p_m * (vt/integ)
            low_bin = bin_to_log_grid(fr_m, p_m, bin_edges)
            combined = intra_med.copy()
            combined[low_bin_mask] = low_bin[low_bin_mask]
            psd_mats['m4_intra_LSmeans'][i, :] = combined
            raw_low['m4_intra_LSmeans'].append((fr_m, p_m))
            f_sp, p_sp = get_sparse_high_freq(d['seasons'], f_cut)
            raw_high['m4_intra_LSmeans'].append((f_sp, p_sp))
        else:
            raw_low['m4_intra_LSmeans'].append((np.array([]), np.array([])))
            raw_high['m4_intra_LSmeans'].append((np.array([]), np.array([])))
    except Exception:
        raw_low['m4_intra_LSmeans'].append((np.array([]), np.array([])))
        raw_high['m4_intra_LSmeans'].append((np.array([]), np.array([])))

# ----------------------------- GLOBAL ENSEMBLE ------------------------------
print("\nGlobal fits...")
global_fits = {}; global_med = {}; global_err = {}; global_N = {}
for m in METHODS:
    mat = psd_mats[m]
    N_c = np.sum(np.isfinite(mat), axis=0)
    mask_b = N_c < MIN_CONTRIB_LEAKAGE
    med = np.nanmedian(mat, axis=0)
    std = np.nanstd(mat, axis=0)
    std_err = std/np.sqrt(np.maximum(N_c, 1))
    med[mask_b] = np.nan; std_err[mask_b] = np.nan
    N_c_masked = N_c.astype(float); N_c_masked[mask_b] = np.nan
    global_med[m] = med; global_err[m] = std_err; global_N[m] = N_c_masked
    global_fits[m] = fit_bpl(bin_centers, med, sigma=std_err)

# ----------------------------- EXAMPLE OBJECT -------------------------------
n_pts_arr = np.array([len(d['t_rest']) for d in valid_data])
T_arr = np.array([d['T_total'] for d in valid_data])
scores = ((n_pts_arr - np.median(n_pts_arr))/np.std(n_pts_arr))**2 + \
         ((T_arr - np.median(T_arr))/np.std(T_arr))**2
ex_idx = int(np.argmin(scores))
ex_d = valid_data[ex_idx]
print(f"Example: {ex_d['folder']} N={len(ex_d['t_rest'])} T={ex_d['T_total']:.0f}")

ex_fits = {}
for m in METHODS:
    b = psd_mats[m][ex_idx, :]
    good = np.isfinite(b)
    if np.sum(good) >= 5:
        ex_fits[m] = fit_bpl(bin_centers[good], b[good], sigma=None)
    else:
        ex_fits[m] = None

# ----------------------------- PLOT PER METHOD ------------------------------
print("Generating per-method plots...")
for m in METHODS:
    fig, (ax_lc, ax_psd) = plt.subplots(2, 1, figsize=(11, 9),
                                         gridspec_kw={'height_ratios': [1, 1.2]})
    # LC
    ax_lc.plot(ex_d['t_rest'], ex_d['f_norm'], 'o', color='red', ms=3,
               alpha=0.7, label='Original (coadded)')
    if m != 'm1_LS_full':
        for k, s in enumerate(ex_d['seasons']):
            f_plot = np.interp(s['t_grid'], s['t_s'], s['f_s'])
            ax_lc.plot(s['t_grid'], f_plot, '-', color='gold', lw=1.0,
                       alpha=0.6, label='Daily interp' if k == 0 else '')
        if m in ('m3_intra_DCT', 'm4_intra_LSmeans'):
            for k, s in enumerate(ex_d['seasons']):
                ax_lc.plot(s['t_mean_actual'], s['f_mean'], 's', color='navy',
                           ms=8, label='Season mean' if k == 0 else '')
    ax_lc.set_xlabel(r'Time [d] (observed frame)')
    ax_lc.set_ylabel(r'Normalized flux')
    ax_lc.set_title(f"{METHOD_LABELS[m]} - {ex_d['folder']} "
                     f"(N={len(ex_d['t_rest'])}, T={ex_d['T_total']:.0f} d, "
                     f"z={ex_d['z']:.3f})")
    ax_lc.legend(fontsize=8, loc='best')
    ax_lc.grid(ls=':', alpha=0.5)

    # PSD
    if m != 'm1_LS_full':
        fr_ref, p_ref = raw_low['m1_LS_full'][ex_idx]
        if fr_ref is not None and len(fr_ref) > 0:
            ax_psd.loglog(fr_ref, p_ref, ':', color='gray', lw=REF_LW,
                          alpha=REF_ALPHA, label='LS full (ref)', zorder=1)

    # Raw low-freq (solid blue)
    f_low, p_low = raw_low[m][ex_idx]
    if f_low is not None and len(f_low) > 0:
        raw_lbl = 'Raw (LS full)' if m == 'm1_LS_full' else 'Raw (low-freq)'
        ax_psd.loglog(f_low, p_low, '-', color='steelblue', lw=RAW_LW,
                      alpha=RAW_ALPHA, label=raw_lbl, zorder=2)

    # Raw high-freq
    if m != 'm1_LS_full':
        f_h, p_h = raw_high[m][ex_idx]
        if f_h is not None and len(f_h) > 0:
            if m in HIGH_FREQ_SPARSE:
                ax_psd.scatter(f_h, p_h, s=SPARSE_SIZE, color='steelblue',
                               alpha=SPARSE_ALPHA,
                               label='Raw (high-freq, sparse)', zorder=1)
            else:
                ax_psd.loglog(f_h, p_h, '-', color='steelblue', lw=RAW_LW,
                              alpha=RAW_ALPHA,
                              label='Raw (high-freq, fine-grid)', zorder=2)

    # Binned
    b = psd_mats[m][ex_idx, :]
    good = np.isfinite(b)
    ax_psd.loglog(bin_centers[good], b[good], 'ko-', ms=7, lw=1.2,
                  label='Binned PSD', zorder=4)

    fit = ex_fits[m]
    if fit:
        f_th = np.logspace(np.log10(bin_centers[good].min()),
                           np.log10(bin_centers[good].max()), 200)
        p_th = model_bpl(f_th, fit['norm'], fit['nu_b'],
                         fit['alpha_high'], fit['noise'])
        lbl = (f"BPL: $\\alpha_{{\\rm high}}$={fit['alpha_high']:.2f}"
               f"$\\pm${fit['alpha_high_err']:.2f}, "
               f"$\\nu_b$={fit['nu_b']:.2e}")
        if fit['at_bound']:
            lbl += f" [{fit['which_bound']}]"
        ax_psd.loglog(f_th, p_th, '--', color=COLORS[m], lw=FIT_LW, label=lbl, zorder=3)
        ax_psd.axvline(fit['nu_b'], color=COLORS[m], ls=':', lw=1.5,
                       alpha=0.8, label=f"$\\nu_b$ = {fit['nu_b']:.2e}")

    ax_psd.set_xlabel(r'$f$ [day$^{-1}$] (observed frame)')
    ax_psd.set_ylabel(r'$P(f)$ [day]')
    ax_psd.set_title(f'PSD - {METHOD_LABELS[m]}')
    ax_psd.legend(fontsize=8, loc='best')
    ax_psd.grid(ls=':', alpha=0.5, which='both')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f'example_{m}.png'), dpi=150)
    plt.close()

# ----------------------------- GLOBAL PLOT ----------------------------------
fig, ax = plt.subplots(figsize=(11, 7))
for m in METHODS:
    med = global_med[m]; err = global_err[m]; N_c = global_N[m]
    good = np.isfinite(med) & np.isfinite(err) & np.isfinite(N_c)
    if not np.any(good):
        continue
    x_g = bin_centers[good]; med_g = med[good]; err_g = err[good]; N_g = N_c[good]
    has_band = N_g >= MIN_BAND_N
    lower = np.maximum(med_g - err_g, med_g * BAND_FLOOR)
    upper = med_g + err_g
    draw_band_and_markers(ax, x_g, med_g, has_band, lower, upper,
                          COLORS[m], METHOD_LABELS[m])
    fit = global_fits[m]
    if fit:
        f_th = np.logspace(np.log10(x_g.min()), np.log10(x_g.max()), 300)
        p_th = model_bpl(f_th, fit['norm'], fit['nu_b'], fit['alpha_high'], fit['noise'])
        ax.loglog(f_th, p_th, '--', color=COLORS[m], lw=1.5,
                  label=f"$\\alpha_{{\\rm high}}$={fit['alpha_high']:.2f}"
                        f"$\\pm${fit['alpha_high_err']:.2f}, "
                        f"$\\nu_b$={fit['nu_b']:.2e}")
        ax.axvline(fit['nu_b'], color=COLORS[m], ls=':', lw=1.2, alpha=0.8)

ax.set_xlabel(r'$f$ [day$^{-1}$] (observed frame)')
ax.set_ylabel(r'$P(f)$ [day]')
ax.set_title(f'Global PSD - all 4 methods (N={len(valid_data)})')
ax.legend(fontsize=7, loc='lower left')
ax.grid(ls=':', alpha=0.5, which='both')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'global_all_methods.png'), dpi=150)
plt.close()

# ----------------------------- F_CUT HISTOGRAM ------------------------------
print("F_cut histogram...")
f_cut_valid = f_cut_arr[np.isfinite(f_cut_arr)]
if len(f_cut_valid) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(f_cut_valid, bins=25, color='steelblue', alpha=0.7, edgecolor='black')
    ax.axvline(np.median(f_cut_valid), color='red', linestyle='--', lw=2,
               label=f'median = {np.median(f_cut_valid):.2e} d$^{{-1}}$')
    ax.set_xlabel(r'$f_{\rm cut}$ [day$^{-1}$] (observed frame)')
    ax.set_ylabel('Number of objects')
    ax.set_title(f'Distribution of $f_{{\\rm cut}}$ (shared by M2, M3, M4) — N={len(f_cut_valid)}')
    ax.set_xscale('log')
    ax.legend(fontsize=10)
    ax.grid(ls=':', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'hist_f_cut.png'), dpi=150)
    plt.close()

# ----------------------------- ENSEMBLES ------------------------------------
logT = 0.25*(np.array([d['LOGE'] for d in valid_data]) -
             np.array([d['LOGM'] for d in valid_data]))
LOGM_arr = np.array([d['LOGM'] for d in valid_data])
LOGE_arr = np.array([d['LOGE'] for d in valid_data])
mi_all = np.digitize(LOGM_arr, mass_edges) - 1
ei_all = np.digitize(LOGE_arr, edd_edges) - 1


def ensemble_plot(method, grouping, edges, labels):
    mat = psd_mats[method]
    groups = []

    if grouping == 'mass':
        n_bins_g = len(mass_edges) - 1
        cmap = plt.get_cmap('viridis')
        shown = []
        for k in range(n_bins_g):
            m_k = (mi_all == k) & (mi_all >= 0) & (mi_all < n_bins_g)
            if int(np.sum(m_k)) >= MIN_OBJS_PER_BIN:
                shown.append(k)
        for i, k in enumerate(shown):
            m_k = (mi_all == k) & (mi_all >= 0) & (mi_all < n_bins_g)
            groups.append({'key': k,
                           'label': f'{mass_edges[k]:.1f} – {mass_edges[k+1]:.1f}',
                           'mask': m_k,
                           'color': cmap(i / max(1, len(shown)-1)),
                           'cells': [(row, k) for row in range(len(edd_edges)-1)]})
    elif grouping == 'edd':
        n_bins_g = len(edd_edges) - 1
        cmap = plt.get_cmap('viridis')
        shown = []
        for k in range(n_bins_g):
            m_k = (ei_all == k) & (ei_all >= 0) & (ei_all < n_bins_g)
            if int(np.sum(m_k)) >= MIN_OBJS_PER_BIN:
                shown.append(k)
        for i, k in enumerate(shown):
            m_k = (ei_all == k) & (ei_all >= 0) & (ei_all < n_bins_g)
            groups.append({'key': k,
                           'label': f'{edd_edges[k]:.1f} – {edd_edges[k+1]:.1f}',
                           'mask': m_k,
                           'color': cmap(i / max(1, len(shown)-1)),
                           'cells': [(k, col) for col in range(len(mass_edges)-1)]})
    else:  # tmax
        cmap = plt.get_cmap('tab10')
        for i, d in enumerate(TMAX_DIAGONALS):
            cells = cells_on_diagonal(d)
            m_k = mask_for_cells(cells)
            if int(np.sum(m_k)) < MIN_OBJS_PER_BIN:
                continue
            groups.append({'key': i,
                           'label': diag_label(d['k_const']),
                           'mask': m_k,
                           'color': cmap(i % cmap.N),
                           'cells': cells,
                           'diag': d})

    fig, (ax_grid, ax_psd) = plt.subplots(
        1, 2, figsize=(18, 6.5),
        gridspec_kw={'width_ratios': [1.0, 1.4]})

    # LEFT: grid
    ax_grid.scatter(LOGM_arr, LOGE_arr, s=3, color='lightgray', alpha=0.35, zorder=1)
    for g in groups:
        m_g = g['mask']
        if int(np.sum(m_g)) == 0:
            continue
        ax_grid.scatter(LOGM_arr[m_g], LOGE_arr[m_g], s=10, color=g['color'],
                        alpha=0.75, zorder=2, edgecolor='none')

    n_rows = len(edd_edges) - 1
    n_cols = len(mass_edges) - 1
    counts_cell = np.zeros((n_rows, n_cols), dtype=int)
    valid_cell = (mi_all >= 0) & (mi_all < n_cols) & (ei_all >= 0) & (ei_all < n_rows)
    for mi, ei in zip(mi_all[valid_cell], ei_all[valid_cell]):
        counts_cell[ei, mi] += 1

    for row in range(n_rows):
        for col in range(n_cols):
            n_cell = int(counts_cell[row, col])
            if n_cell == 0:
                continue
            x_c = (mass_edges[col] + mass_edges[col+1]) / 2
            y_c = (edd_edges[row] + edd_edges[row+1]) / 2
            txt = ax_grid.text(x_c, y_c, str(n_cell), ha='center', va='center',
                                fontsize=8, fontweight='bold', color='white', zorder=5)
            txt.set_path_effects([patheffects.withStroke(linewidth=2.0, foreground='black')])

    for e in mass_edges:
        ax_grid.axvline(e, color='k', lw=0.6, ls=':', alpha=0.6)
    for e in edd_edges:
        ax_grid.axhline(e, color='k', lw=0.6, ls=':', alpha=0.6)

    if grouping == 'tmax':
        for g in groups:
            cells = g['cells']
            if len(cells) < 2:
                continue
            row_s, col_s = cells[0]; row_e, col_e = cells[-1]
            x_s = (mass_edges[col_s] + mass_edges[col_s+1]) / 2
            y_s = (edd_edges[row_s] + edd_edges[row_s+1]) / 2
            x_e = (mass_edges[col_e] + mass_edges[col_e+1]) / 2
            y_e = (edd_edges[row_e] + edd_edges[row_e+1]) / 2
            ax_grid.plot([x_s, x_e], [y_s, y_e], '-', color=g['color'],
                          lw=2.0, alpha=0.9, zorder=4)

    ax_grid.set_xlim(mass_edges[0], mass_edges[-1])
    ax_grid.set_ylim(edd_edges[0], edd_edges[-1])
    ax_grid.set_xticks(mass_edges)
    ax_grid.set_yticks(edd_edges)
    ax_grid.set_xlabel(r'$\log M_{\rm BH}$')
    ax_grid.set_ylabel(r'$\log \lambda_{\rm Edd}$')
    ax_grid.set_title(f'Grid colored by {grouping}')

    # RIGHT: PSD with fit params in legend
    res = []
    for g in groups:
        m_g = g['mask']
        idxs = np.where(m_g)[0]
        if len(idxs) < MIN_OBJS_PER_BIN:
            continue
        sub = mat[idxs, :]
        N_c = np.sum(np.isfinite(sub), axis=0)
        good = N_c >= MIN_CONTRIB_ENSEMBLE
        if np.sum(good) < 4:
            continue
        med_k = np.nanmedian(sub[:, good], axis=0)
        std_k = np.nanstd(sub[:, good], axis=0)
        N_good = N_c[good]
        err_k = std_k / np.sqrt(np.maximum(N_good, 1))
        x = bin_centers[good]
        color = g['color']
        has_band = N_good >= MIN_BAND_N
        lower = np.maximum(med_k - err_k, med_k * BAND_FLOOR)
        upper = med_k + err_k

        fit = fit_bpl(x, med_k, sigma=err_k)
        legend_lbl = legend_with_fit(g['label'], len(idxs), fit)

        draw_band_and_markers(ax_psd, x, med_k, has_band, lower, upper,
                              color, legend_lbl, ms=5, ms_open=7)

        if fit:
            f_th = np.logspace(np.log10(x.min()), np.log10(x.max()), 200)
            p_th = model_bpl(f_th, fit['norm'], fit['nu_b'],
                             fit['alpha_high'], fit['noise'])
            ax_psd.loglog(f_th, p_th, '--', color=color, lw=1.3, alpha=0.9)
            ax_psd.axvline(fit['nu_b'], color=color, ls=':', lw=1.1, alpha=0.7)
            if grouping == 'tmax':
                center_val = np.nanmedian(logT[idxs])
            elif grouping == 'mass':
                center_val = np.median(LOGM_arr[idxs])
            else:
                center_val = np.median(LOGE_arr[idxs])
            lowN_bins = x[~has_band]
            lowN_str = (f"[{np.min(lowN_bins):.1e}, {np.max(lowN_bins):.1e}]"
                        if len(lowN_bins) > 0 else "none")
            res.append({'method': method, 'group': grouping, 'bin': g['key'],
                        'label': g['label'], 'N': len(idxs), 'center': center_val,
                        'norm': fit['norm'], 'nu_b': fit['nu_b'],
                        'nu_b_err': fit['nu_b_err'],
                        'alpha_high': fit['alpha_high'],
                        'alpha_high_err': fit['alpha_high_err'],
                        'at_bound': fit['at_bound'],
                        'which_bound': fit['which_bound'],
                        'bic': fit['bic'], 'chi2': fit['chi2'],
                        'N_pts': fit['N_pts'],
                        'N_bin_median': int(np.median(N_good)),
                        'N_bin_min': int(np.min(N_good)),
                        'f_at_Nmin': float(x[np.argmin(N_good)]),
                        'n_lowN_bins': int(np.sum(~has_band)),
                        'f_lowN_range': lowN_str})

    ax_psd.set_xlabel(r'$f$ [day$^{-1}$] (observed frame)')
    ax_psd.set_ylabel(r'$P(f)$ [day]')
    ax_psd.set_title(f'{METHOD_LABELS[method]} — {grouping}')
    ax_psd.legend(fontsize=7, loc='lower left', framealpha=0.9)
    ax_psd.grid(ls=':', alpha=0.5, which='both')

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f'ensemble_{grouping}_{method}.png'), dpi=150)
    plt.close()
    return res

print("Ensembles...")
mass_labels = [f'{mass_edges[i]:.1f} – {mass_edges[i+1]:.1f}'
               for i in range(len(mass_edges)-1)]
edd_labels = [f'{edd_edges[i]:.1f} – {edd_edges[i+1]:.1f}'
              for i in range(len(edd_edges)-1)]

ens_results = []
for m in METHODS:
    for grp, edges, labs in [('mass', mass_edges, mass_labels),
                              ('edd', edd_edges, edd_labels),
                              ('tmax', None, None)]:
        ens_results.extend(ensemble_plot(m, grp, edges, labs))

df_ens = pd.DataFrame(ens_results)
df_ens.to_csv(os.path.join(PLOTS_DIR, 'ensemble_fits_4methods.csv'), index=False)

# ----------------------------- REPORT ---------------------------------------
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write("4-METHODS COMPARISON (OBSERVED FRAME, no cosmological redshift)\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"N objects: {len(valid_data)}\n")
    f.write(f"Filter: err <= {FLUX_ERR_FACTOR}*|flux|\n")
    f.write(f"Frame: OBSERVED (no /(1+z) correction). Frequencies in d^-1.\n")
    f.write(f"Grid: {n_bins} bins, [{f_min_g:.3e}, {F_MAX_NYQUIST:.3f}] d^-1\n")
    f.write(f"Fine grid (M2 continuous): {N_FINE_GRID} pts\n")
    f.write(f"M3/M4 raw high-freq: SPARSE season DCTs (unbinned scatter)\n")
    f.write(f"MIN_OBJS_PER_BIN = {MIN_OBJS_PER_BIN}\n")
    f.write(f"MIN_BAND_N = {MIN_BAND_N} (bins with N < {MIN_BAND_N} -> open circles)\n")
    f.write(f"BAND_FLOOR = {BAND_FLOOR} (band lower >= {BAND_FLOOR} * median)\n")
    f.write(f"BAND_ALPHA = {BAND_ALPHA}\n")
    f.write(f"mass_edges = {mass_edges}\n")
    f.write(f"edd_edges  = {edd_edges}\n")
    f.write(f"\nGrid convention: x = log M_BH (columns = mass bins),\n")
    f.write(f"                 y = log lambda_Edd (rows = edd bins).\n")
    f.write(f"TMAX_DIAGONALS: {len(TMAX_DIAGONALS)} isotherms (col_mass - row_edd = const)\n")
    for d in TMAX_DIAGONALS:
        k = d['k_const']
        cells = cells_on_diagonal(d)
        f.write(f"  {diag_label(k)}: col-row = {k:+d}, "
                f"cells (row,col) = {cells}, "
                f"N = {int(np.sum(mask_for_cells(cells)))}\n")
    f.write("\n")

    f.write("=== F_CUT DISTRIBUTION ===\n")
    f.write(f"  N valid: {len(f_cut_valid)}\n")
    if len(f_cut_valid) > 0:
        f.write(f"  median = {np.median(f_cut_valid):.3e} d^-1\n")
        f.write(f"  min    = {np.min(f_cut_valid):.3e} d^-1\n")
        f.write(f"  max    = {np.max(f_cut_valid):.3e} d^-1\n")
        f.write(f"  25/75  = [{np.percentile(f_cut_valid,25):.3e}, "
                f"{np.percentile(f_cut_valid,75):.3e}] d^-1\n")
    f.write("\n")

    f.write(f"Example object: {ex_d['folder']}\n")
    f.write(f"  N={len(ex_d['t_rest'])}, T={ex_d['T_total']:.1f} d, z={ex_d['z']:.4f}, "
            f"n_seasons={ex_d['n_seasons']}\n")
    f.write(f"  logM={ex_d['LOGM']:.2f}, logEdd={ex_d['LOGE']:.2f}\n\n")

    f.write("=== EXAMPLE OBJECT FITS ===\n")
    f.write(f"{'Method':<30} {'a_high':>20} {'nu_b':>22} {'BIC':>8}\n")
    f.write("-" * 85 + "\n")
    for m in METHODS:
        fit = ex_fits[m]
        if not fit:
            f.write(f"{m:<30}   fit failed\n"); continue
        a_h = f"{fit['alpha_high']:.2f}+/-{fit['alpha_high_err']:.2f}"
        n_b = f"{fit['nu_b']:.2e}+/-{fit['nu_b_err']:.1e}"
        if fit['at_bound']: n_b += f"[{fit['which_bound']}]"
        f.write(f"{m:<30} {a_h:>20} {n_b:>22} {fit['bic']:>8.2f}\n")
    f.write("\n")

    f.write("=== GLOBAL TABLE ===\n")
    f.write(f"{'Source':<40} {'a_low':>6} {'a_high':>18} {'nu_b (d^-1)':>22}\n")
    f.write("-" * 90 + "\n")
    for src, v in LIT.items():
        a_h = f"{v['a_high']:.1f}+/-{v['a_high_err']:.1f}" if v['a_high'] else "  --  "
        f.write(f"{src:<40} {'-1':>6} {a_h:>18} {v['nu_b']:>10.1e}\n")
    f.write("-" * 90 + "\n")
    for m in METHODS:
        fit = global_fits[m]
        if not fit: continue
        a_h = f"{fit['alpha_high']:.2f}+/-{fit['alpha_high_err']:.2f}"
        n_b = f"{fit['nu_b']:.2e}+/-{fit['nu_b_err']:.1e}"
        if fit['at_bound']: n_b += f"[{fit['which_bound']}]"
        f.write(f"{METHOD_LABELS[m]:<40} {'-1':>6} {a_h:>18} {n_b:>22}\n")
    f.write("\n")

    f.write("=== GLOBAL PER-BIN N DIAGNOSTIC ===\n")
    for m in METHODS:
        N_c = global_N[m]
        good_N = np.isfinite(N_c)
        if not np.any(good_N): continue
        x = bin_centers[good_N]; Nv = N_c[good_N]
        low_mask = Nv < MIN_BAND_N
        f.write(f"\n[{m}]\n")
        f.write(f"  N_bins valid: {len(Nv)}\n")
        f.write(f"  N_contributing: median={np.median(Nv):.0f}, "
                f"min={np.min(Nv):.0f}, max={np.max(Nv):.0f}\n")
        f.write(f"  N_min located at f = {x[np.argmin(Nv)]:.2e} d^-1\n")
        if np.any(low_mask):
            f.write(f"  Bins with N < {MIN_BAND_N}: {np.sum(low_mask)} "
                    f"(f range [{x[low_mask].min():.1e}, {x[low_mask].max():.1e}])\n")
        else:
            f.write(f"  All bins have N >= {MIN_BAND_N}\n")
    f.write("\n")

    f.write("=== BIC COMPARISON (global) ===\n")
    valid_bics = [global_fits[m]['bic'] for m in METHODS if global_fits[m]]
    if valid_bics:
        bic_min = min(valid_bics)
        f.write(f"{'Method':<30} {'k':>3} {'N':>5} {'BIC':>10} {'dBIC':>8}\n")
        f.write("-" * 60 + "\n")
        for m in METHODS:
            fit = global_fits[m]
            if not fit: continue
            f.write(f"{m:<30} {4:>3} {fit['N_pts']:>5} "
                    f"{fit['bic']:>10.2f} {fit['bic']-bic_min:>8.2f}\n")
    f.write("\n")

    f.write("=== BINS DIAGNOSTIC ===\n")
    f.write(f"\n[mass] (columns, N >= {MIN_OBJS_PER_BIN} to be shown)\n")
    for k in range(len(mass_edges)-1):
        m_k = (mi_all == k) & (mi_all >= 0) & (mi_all < len(mass_edges)-1)
        n_k = int(np.sum(m_k))
        status = "SHOWN" if n_k >= MIN_OBJS_PER_BIN else f"SKIPPED (N<{MIN_OBJS_PER_BIN})"
        f.write(f"  col {k}  {mass_edges[k]:.1f} - {mass_edges[k+1]:.1f}   "
                f"N={n_k:>5}   {status}\n")
    f.write(f"\n[edd] (rows, N >= {MIN_OBJS_PER_BIN} to be shown)\n")
    for k in range(len(edd_edges)-1):
        m_k = (ei_all == k) & (ei_all >= 0) & (ei_all < len(edd_edges)-1)
        n_k = int(np.sum(m_k))
        status = "SHOWN" if n_k >= MIN_OBJS_PER_BIN else f"SKIPPED (N<{MIN_OBJS_PER_BIN})"
        f.write(f"  row {k}  {edd_edges[k]:.1f} - {edd_edges[k+1]:.1f}   "
                f"N={n_k:>5}   {status}\n")
    f.write(f"\n[tmax] (diagonais isotermicas; celula = (row_edd, col_mass))\n")
    for i, d in enumerate(TMAX_DIAGONALS):
        cells = cells_on_diagonal(d)
        m_k = mask_for_cells(cells)
        n_k = int(np.sum(m_k))
        status = "SHOWN" if n_k >= MIN_OBJS_PER_BIN else f"SKIPPED (N<{MIN_OBJS_PER_BIN})"
        f.write(f"  {diag_label(d['k_const'])}   N={n_k:>5}   {status}\n")
        f.write(f"     cells (row,col): {cells}\n")
        logT_cells = []
        for row, col in cells:
            x_c = (mass_edges[col] + mass_edges[col+1]) / 2
            y_c = (edd_edges[row] + edd_edges[row+1]) / 2
            logT_cells.append(0.25 * (y_c - x_c))
        f.write(f"     logT(cell centers): "
                + ", ".join(f"{v:.2f}" for v in logT_cells) + "\n")
    f.write("\n")

    f.write("=== ENSEMBLES ===\n")
    for m in METHODS:
        f.write(f"\n[{m}]\n")
        f.write(f"{'group':<6} {'label':<14} {'N':>5} {'a_high':>18} "
                f"{'nu_b':>22} {'at_bound':>10} {'Nmed':>5} {'Nmin':>5} {'f@Nmin':>10}\n")
        f.write("-" * 115 + "\n")
        sub = df_ens[df_ens['method'] == m]
        for _, r in sub.iterrows():
            a_h = f"{r['alpha_high']:.2f}+/-{r['alpha_high_err']:.2f}"
            n_b = f"{r['nu_b']:.2e}+/-{r['nu_b_err']:.1e}"
            if r['at_bound']: n_b += f"[{r['which_bound']}]"
            f.write(f"{r['group']:<6} {str(r['label']):<14} {r['N']:>5} "
                    f"{a_h:>18} {n_b:>22} {str(r['at_bound']):>10} "
                    f"{r['N_bin_median']:>5} {r['N_bin_min']:>5} "
                    f"{r['f_at_Nmin']:>10.2e}\n")
        f.write(f"\n  Low-N bins (N < {MIN_BAND_N}):\n")
        for _, r in sub.iterrows():
            f.write(f"    {r['label']:<14}: {r['n_lowN_bins']} bins, "
                    f"f range = {r['f_lowN_range']}\n")
    f.write("\n")

print(f"\nReport: {REPORT_PATH}")
print(f"CSV: {os.path.join(PLOTS_DIR, 'ensemble_fits_4methods.csv')}")
print(f"F_cut histogram: {os.path.join(PLOTS_DIR, 'hist_f_cut.png')}")
print(f"Plots: {PLOTS_DIR}/")
# END OF FILE

Loading metadata...


Crossmatch: 100%|██████████| 2261/2261 [00:00<00:00, 30163.52it/s]


Matched: 468


Processing: 100%|██████████| 468/468 [00:07<00:00, 64.93it/s]


Valid: 468
Grid: 31 bins, [3.920e-04, 0.500] d^-1 (OBSERVED frame)
Fine grid: 300 pts


Methods: 100%|██████████| 468/468 [00:01<00:00, 253.90it/s]



Global fits...
Example: 358.6566099_18.0898832 N=439 T=2243
Generating per-method plots...
F_cut histogram...
Ensembles...

Report: plots_4methods_g\report_4methods.txt
CSV: plots_4methods_g\ensemble_fits_4methods.csv
F_cut histogram: plots_4methods_g\hist_f_cut.png
Plots: plots_4methods_g/
